# Data to Metadata to Dummy Data

In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv(
    "penguin_plus.csv", 
    parse_dates=["timestamp", "timestamp_with_time"]
)
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750.0,True,73,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800.0,False,1,2025-12-15,2025-12-15 18:15:26,2


In [3]:
import sys
sys.path.append('../')

In [4]:
from csvw_safe.make_metadata_from_data import make_metadata_from_data
from csvw_safe.validate_metadata import validate_metadata
from csvw_safe.validate_metadata_shacl import validate_metadata_shacl
from csvw_safe.make_dummy_from_metadata import make_dummy_from_metadata
from csvw_safe.assert_same_structure import assert_same_structure

In [5]:
shacl_path = '../../csvw-safe-constraints.ttl'

In [6]:
metadata_path_1 = 'metadata/penguin_metadata_basic.json-ld'

metadata_path_2 = 'metadata/penguin_metadata_continuous_partitions.json-ld'
metadata_path_3 = 'metadata/penguin_metadata_continuous_partitions_column-contrib.json-ld'
metadata_path_4 = 'metadata/penguin_metadata_continuous_partitions_partition-contrib.json-ld'

metadata_path_5 = 'metadata/penguin_metadata_column_groups.json-ld'

metadata_path_6 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups.json-ld'
metadata_path_7 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups_column-contrib.json-ld'
metadata_path_8 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups_partition-contrib.json-ld'

## Basic

In [7]:
metadata_1 = make_metadata_from_data(
    df, privacy_unit = "penguin_id"
)
with open(metadata_path_1, 'w', encoding='utf-8') as f:
    json.dump(metadata_1, f)

In [8]:
validate_metadata(metadata_1)
validate_metadata_shacl(metadata_path_1, shacl_path)

(True, 'Validation Report\nConforms: True\n')

In [9]:
dummy_df_1 = make_dummy_from_metadata(metadata_1, nb_rows = 100, seed = 0)
dummy_df_1.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,i,e,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,g,h,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [10]:
assert_same_structure(df, dummy_df_1, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## Known continuous partitions

In [11]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}

In [12]:
metadata_2 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions
)
with open(metadata_path_2, 'w', encoding='utf-8') as f:
    json.dump(metadata_2, f)

In [13]:
validate_metadata(metadata_2)
validate_metadata_shacl(metadata_path_2, shacl_path)

(False,
 'Validation Report\nConforms: False\nResults (10):\nConstraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):\n\tSeverity: sh:Violation\n\tSource Shape: [ sh:datatype xsd:integer ; sh:minInclusive Literal("0", datatype=xsd:integer) ; sh:path csvw-safe:partitions ]\n\tFocus Node: [ csvw-safe:exhaustivePartitions Literal("true" = True, datatype=xsd:boolean) ; csvw-safe:maxNumPartitions Literal("2", datatype=xsd:integer) ; csvw-safe:partitions [ csvw-safe:dp.maxContributions Literal("2", datatype=xsd:integer) ; csvw-safe:dp.maxGroupsPerUnit Literal("3", datatype=xsd:integer) ; csvw-safe:dp.maxLength Literal("139", datatype=xsd:integer) ; csvw-safe:part.predicate [ csvw-safe:part.lowerBound Literal("2025-01-02T00:00:00") ; csvw-safe:part.upperBound Literal("2025-06-02T00:00:00") ; rdf:type rdfs:Resource ] ; rdf:type csvw-safe:Partition, rdfs:Resource ], [ csvw-safe:dp.maxContributions Literal("2", datatype=xsd:integer) ; csvw-saf

In [14]:
dummy_df_2 = make_dummy_from_metadata(metadata_2, nb_rows = 100, seed = 0)
dummy_df_2.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,i,e,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,g,h,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [15]:
assert_same_structure(df, dummy_df_2, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## With DP at column level

In [16]:
metadata_3 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    default_contributions_level = 'column'
)
with open(metadata_path_3, 'w', encoding='utf-8') as f:
    json.dump(metadata_3, f)

In [17]:
validate_metadata(metadata_3)
validate_metadata_shacl(metadata_path_3, shacl_path)

(False,
 'Validation Report\nConforms: False\nResults (26):\nConstraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):\n\tSeverity: sh:Violation\n\tSource Shape: [ sh:datatype xsd:integer ; sh:minInclusive Literal("0", datatype=xsd:integer) ; sh:path csvw-safe:partitions ]\n\tFocus Node: [ csvw-safe:dp.maxContributions Literal("1", datatype=xsd:integer) ; csvw-safe:dp.maxGroupsPerUnit Literal("3", datatype=xsd:integer) ; csvw-safe:dp.maxLength Literal("152", datatype=xsd:integer) ; csvw-safe:exhaustivePartitions Literal("true" = True, datatype=xsd:boolean) ; csvw-safe:maxNumPartitions Literal("3", datatype=xsd:integer) ; csvw-safe:partitions Literal("Adelie"), Literal("Chinstrap"), Literal("Gentoo") ; csvw-safe:privacyId Literal("false" = False, datatype=xsd:boolean) ; csvw-safe:synth.nullableProportion Literal("0.0", datatype=xsd:double) ; csvw:datatype Literal("string") ; csvw:name Literal("species") ; csvw:required Literal("true" =

In [18]:
dummy_df_3 = make_dummy_from_metadata(metadata_3, nb_rows = 100, seed = 0)
dummy_df_3.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [20]:
assert_same_structure(df, dummy_df_3, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## With DP at partition level

In [21]:
metadata_4 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    default_contributions_level = 'partition'
)
with open(metadata_path_4, 'w', encoding='utf-8') as f:
    json.dump(metadata_4, f)

In [22]:
validate_metadata(metadata_4)
validate_metadata_shacl(metadata_path_4, shacl_path)

(False,
 'Validation Report\nConforms: False\nResults (38):\nConstraint Violation in DatatypeConstraintComponent (http://www.w3.org/ns/shacl#DatatypeConstraintComponent):\n\tSeverity: sh:Violation\n\tSource Shape: [ sh:datatype xsd:integer ; sh:minInclusive Literal("0", datatype=xsd:integer) ; sh:path csvw-safe:partitions ]\n\tFocus Node: [ csvw-safe:exhaustivePartitions Literal("true" = True, datatype=xsd:boolean) ; csvw-safe:maxNumPartitions Literal("2", datatype=xsd:integer) ; csvw-safe:partitions [ csvw-safe:dp.maxContributions Literal("1", datatype=xsd:integer) ; csvw-safe:dp.maxGroupsPerUnit Literal("3", datatype=xsd:integer) ; csvw-safe:dp.maxLength Literal("165", datatype=xsd:integer) ; csvw-safe:part.predicate [ csvw-safe:part.partitionValue Literal("false" = False, datatype=xsd:boolean) ; rdf:type rdfs:Resource ] ; rdf:type csvw-safe:Partition, rdfs:Resource ], [ csvw-safe:dp.maxContributions Literal("1", datatype=xsd:integer) ; csvw-safe:dp.maxGroupsPerUnit Literal("3", data

In [23]:
dummy_df_4 = make_dummy_from_metadata(metadata_4, nb_rows = 100, seed = 0)
dummy_df_4.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [24]:
assert_same_structure(df, dummy_df_4, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## Known column groups

In [25]:
column_groups = [
    ["species", "island"],
    ["species", "island", "favourite_number"]
]

In [26]:
metadata_5 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_5, 'w', encoding='utf-8') as f:
    json.dump(metadata_5, f)

In [27]:
validate_metadata(metadata_5)
validate_metadata_shacl(metadata_path_5, shacl_path)

ValueError: ColumnGroup references unknown column 'species'

In [28]:
dummy_df_5 = make_dummy_from_metadata(metadata_5, nb_rows = 100, seed = 0)
dummy_df_5.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [29]:
assert_same_structure(df, dummy_df_5, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## Known column groups containing public continuous partitions

In [30]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}
column_groups = [
    ["species", "island"],
    ["species", "island", "bill_length_mm"],
]

In [31]:
metadata_6 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_6, 'w', encoding='utf-8') as f:
    json.dump(metadata_6, f)

In [32]:
validate_metadata(metadata_6)
validate_metadata_shacl(metadata_path_6, shacl_path)

ValueError: ColumnGroup references unknown column 'species'

In [33]:
dummy_df_6 = make_dummy_from_metadata(metadata_6, nb_rows = 100, seed = 0)
dummy_df_6.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [34]:
assert_same_structure(df, dummy_df_6, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## Known column groups containing public continuous partitions - dp contrib column

In [35]:
metadata_7 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'column'
)
with open(metadata_path_7, 'w', encoding='utf-8') as f:
    json.dump(metadata_7, f)

In [36]:
validate_metadata(metadata_7)
validate_metadata_shacl(metadata_path_7, shacl_path)

ValueError: ColumnGroup references unknown column 'species'

In [37]:
dummy_df_7 = make_dummy_from_metadata(metadata_7, nb_rows = 100, seed = 0)
dummy_df_7.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [38]:
assert_same_structure(df, dummy_df_7, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.


## Known column groups containing public continuous partitions - dp contrib partition

In [39]:
metadata_8 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups,
    default_contributions_level = 'partition'
)
with open(metadata_path_8, 'w', encoding='utf-8') as f:
    json.dump(metadata_8, f)

In [40]:
validate_metadata(metadata_8)
validate_metadata_shacl(metadata_path_8, shacl_path)

ValueError: ColumnGroup references unknown column 'species'

In [41]:
dummy_df_8 = make_dummy_from_metadata(metadata_8, nb_rows = 100, seed = 0)
dummy_df_8.head(2)

species
island
bill_length_mm
bill_depth_mm
flipper_length_mm
body_mass_g
sex
penguin_id
timestamp
timestamp_with_time
favourite_number


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,NaN,3515.0,False,60,2025-01-25,2025-06-16 06:47:46,4
1,Chinstrap,Torgersen,38.490255,19.071391,194.0,3668.0,False,49,2025-03-21,2025-02-20 06:47:46,5


In [42]:
assert_same_structure(df, dummy_df_8, check_categories = False)

Structure check passed: 11 columns match, datatypes compatible, nullability compatible.
